# Assignment 4 (new data) — 3. MNIST (CNN)

**One learning problem, three implementations: NumPy from scratch, TensorFlow/Keras, and PyTorch.**

| # | Dataset | Structure | Model family |
|---|---------|-----------|--------------|
| A | BRFSS diabetes | tabular, 19 features | MLP |
| B | Rice images | image, 3×32×32 | CNN |
| **C** | **MNIST** | **image, 1×28×28** | **CNN** |

### Why MNIST is here

The third dataset was meant to be Intel Image Classification. It cannot be used:
`00_inventory.ipynb` establishes that `data/` holds only `seg_pred/` — 7,301 loose files with no class
sub-folders — while the labelled `seg_train/` and `seg_test/` splits are absent. In that dataset the folder
name *is* the label, so there is no ground truth to train against or score with.

MNIST stands in, downloaded automatically by `torchvision` on first run. It is the natural substitute:

- It is genuinely a **second image dataset** with different properties from the rice grains — ten classes
  instead of five, **greyscale** (1 channel) instead of RGB, handwriting instead of photography.
- It is the smallest of the image problems, so the from-scratch leg is comfortable on it.
- The repository root already ran MNIST in `02_mnist.ipynb` with this exact architecture. That turns
  §7 of this notebook into something the Intel data could never have offered: an **independent
  reproduction check**, rerunning a committed experiment and asking whether the same numbers come back.

### The architecture is deliberately identical to `02_mnist.ipynb`

Same layer sizes, same 20,490 parameters, same subset sizes, same hyperparameters, same seed. Changing
anything would make §7 meaningless.

$$
X_{1\times28\times28}
\rightarrow \underbrace{\text{Conv}_{1\rightarrow16} \rightarrow \text{ReLU} \rightarrow \text{MaxPool}_2}_{14\times14}
\rightarrow \underbrace{\text{Conv}_{16\rightarrow32} \rightarrow \text{ReLU} \rightarrow \text{MaxPool}_2}_{7\times7}
\rightarrow \text{Flatten}_{1568} \rightarrow \text{Dense}_{10}
$$

Note there is **no hidden dense layer** here — the flattened features go straight to the ten logits. That
is the tutorial's own MNIST design, and it makes the parameter split even more lopsided than the rice model:
see §2.

### The fairness rule

> Same Dataset **+** Same Split **+** Same Architecture **+** Comparable Hyperparameters

All three legs pull their arrays from one `ass4_utils.load_mnist` call with one seed.

In [ ]:
import os, sys, json, platform

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath("."))        # notebook/  -> ass4_newdata
sys.path.insert(0, os.path.abspath(".."))       # repo root  -> ass4_utils, scratch_nn

import ass4_newdata as D
import ass4_utils as U
import scratch_nn as S

U.set_seed(U.SEED)

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import tensorflow as tf
import keras

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Outputs stay inside notebook/: weights in notebook/models/, metrics in
# notebook/results/. The six original notebooks keep theirs under the repo-root
# results/, so nothing here can overwrite them.
RESULTS = D.notebook_results_dir()
MODELS = D.use_notebook_model_dir()
print("weights ->", os.path.relpath(MODELS, D.ROOT))
print("metrics ->", os.path.relpath(RESULTS, D.ROOT))
plt.rcParams["figure.dpi"] = 110

print("torch     ", torch.__version__, "| device:", DEVICE,
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")
print("tensorflow", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU") or "none (CPU)")

---
## 1. The data

MNIST is not shipped in `data/` — `ass4_utils.load_mnist` fetches it through `torchvision` on first run
(about 12 MB) into `data/mnist/`, and reuses it afterwards. That is the same loader, with the same seed
and the same normalisation, that `02_mnist.ipynb` used, which is what makes §7 a fair comparison.

The three-way comparison runs on a **10,000-image subset**, because the NumPy leg is roughly an order of
magnitude slower than the framework legs. Keras and PyTorch are then rerun on the **full 60,000**.

In [ ]:
N_TRAIN_SUB, N_TEST_SUB = 10_000, 2_000
EPOCHS, BATCH_SIZE, LR = 5, 64, 1e-3

X_train, y_train, X_test, y_test, meta = U.load_mnist(
    n_train=N_TRAIN_SUB, n_test=N_TEST_SUB)

C, H, W = meta["shape"]
CLASSES = meta["classes"]
N_CLASSES = len(CLASSES)
DATASET_SUB = "MNIST (10k subset)"

print(f"\ntrain {X_train.shape}   test {X_test.shape}   dtype {X_train.dtype}")
print(f"input shape (C,H,W) = {(C, H, W)}   classes = {N_CLASSES}")
print(f"shared hyperparameters: epochs={EPOCHS}, batch_size={BATCH_SIZE}, optimizer=Adam(lr={LR})")
print(f"\nper-class train counts: {np.bincount(y_train)}")

In [ ]:
U.plot_samples(X_train, y_train, CLASSES, n=12,
               title=f"{DATASET_SUB} - 28x28 greyscale inputs as the network sees them")
plt.show()

> **One channel, not three.** This is the only image dataset in the collection with `C = 1`. Everything in
> `scratch_nn.Conv2D` already handles it — the kernel window flattens to
> $C \cdot K_h \cdot K_w = 1 \cdot 3 \cdot 3 = 9$ values per row instead of the 27 the rice images needed —
> but it is the reason the first convolution here costs 160 parameters where the rice model's cost 896.

---
## 2. The shared architecture

### Counting the parameters by hand first

Tutorial §43:

$$N_{\text{conv}} = (C_{\text{in}} \cdot K_h \cdot K_w) \cdot C_{\text{out}} + C_{\text{out}}
\qquad
N_{\text{linear}} = N_{\text{in}} N_{\text{out}} + N_{\text{out}}$$

In [ ]:
CH1, CH2, K, PAD = 16, 32, 3, 1
FLAT = CH2 * (H // 4) * (W // 4)

manual = [
    (f"Conv2D({C}->{CH1}, k{K})",    CH1 * C   * K * K + CH1),
    ("ReLU",                         0),
    ("MaxPool2D(2)",                 0),
    (f"Conv2D({CH1}->{CH2}, k{K})",  CH2 * CH1 * K * K + CH2),
    ("ReLU",                         0),
    ("MaxPool2D(2)",                 0),
    ("Flatten",                      0),
    (f"Linear({FLAT}->{N_CLASSES})", FLAT * N_CLASSES + N_CLASSES),
]
EXPECTED_PARAMS = sum(p for _, p in manual)
ARCH = "CNN 2conv+fc"

print(f"{'layer':<26}{'params':>10}")
print("-" * 36)
for name, p in manual:
    print(f"{name:<26}{p:>10,}")
print("-" * 36)
print(f"{'TOTAL (by hand)':<26}{EXPECTED_PARAMS:>10,}")

head = FLAT * N_CLASSES + N_CLASSES
conv = CH1 * C * K * K + CH1 + CH2 * CH1 * K * K + CH2
print(f"\nflattened features: {CH2} * {H//4} * {W//4} = {FLAT}")
print(f"the single Linear layer holds {head:,} of {EXPECTED_PARAMS:,} parameters "
      f"({head / EXPECTED_PARAMS:.0%}), while the two convolutions that do the actual "
      f"feature extraction hold only {conv:,} ({conv / EXPECTED_PARAMS:.0%}).")
print(f"\nmatches 02_mnist.ipynb's committed count of 20,490? {EXPECTED_PARAMS == 20490}")

results = []        # the 10k-subset three-way comparison
full_results = []   # the full-60k Keras/PyTorch runs

> **The same lopsidedness as every other CNN here.** The convolutions cost 4,800 parameters and do the
> visual work; the one dense layer that reads their output costs 15,690. Notebook 04's M1–M4 models replace
> that head with global average pooling for exactly this reason.

---
## 3. Implementation A — from scratch (NumPy)

In [ ]:
U.set_seed(U.SEED)

def build_scratch_cnn():
    return S.Sequential([
        S.Conv2D(C, CH1, k=K, stride=1, pad=PAD, seed=1),
        S.ReLU(),
        S.MaxPool2D(2, 2),
        S.Conv2D(CH1, CH2, k=K, stride=1, pad=PAD, seed=2),
        S.ReLU(),
        S.MaxPool2D(2, 2),
        S.Flatten(),
        S.Dense(FLAT, N_CLASSES, seed=3),
    ])

scratch_model = build_scratch_cnn()
print(scratch_model.summary(input_shape=(C, H, W)))
print(f"\nmatches hand count {EXPECTED_PARAMS:,}? {scratch_model.n_params() == EXPECTED_PARAMS}")

### Are the hand-written gradients actually right?

Checked against central finite differences

$$\frac{\partial \mathcal{L}}{\partial \theta_i} \approx \frac{\mathcal{L}(\theta_i + \epsilon) - \mathcal{L}(\theta_i - \epsilon)}{2\epsilon}$$

on a tiny model in float64.

**Why the step size is 1e-5 and not the default 1e-3.** `MaxPool2D` routes the gradient to the largest
element of each window, so the loss is only piecewise differentiable: where two elements tie, an
epsilon-sized nudge changes which one wins, and the numerical derivative then measures a different branch
from the one the analytic gradient took. MNIST digits sit on a large black background, so the convolution
output is constant over big regions and ties are common — the same effect notebook 02 measures on the rice
images, where about 65 % of pooling windows contain one. A coarse probe reports a large error on a backward
pass that is in fact correct. The MLP in notebook 01 has no pooling layer and passes at the default step.

In [ ]:
U.set_seed(U.SEED)
check_model = S.Sequential([
    S.Conv2D(C, 4, k=3, stride=1, pad=1, seed=1),
    S.ReLU(),
    S.MaxPool2D(2, 2),
    S.Flatten(),
    S.Dense(4 * (H // 2) * (W // 2), N_CLASSES, seed=2),
])
# eps=1e-5, not the 1e-3 default: see the note above on max-pool ties.
err = S.gradient_check(check_model, X_train[:4], y_train[:4], n_samples=6, eps=1e-5)
print(f"worst relative error vs finite differences: {err:.3e}")
print("PASS - hand-derived gradients agree with the numerical derivative" if err < 1e-4
      else "FAIL - backward pass disagrees with finite differences")

In [ ]:
U.set_seed(U.SEED)
with U.Timer() as t_scratch:
    hist_scratch = S.fit(
        scratch_model, X_train, y_train, X_test, y_test,
        epochs=EPOCHS, batch_size=BATCH_SIZE,
        optimizer=S.Adam(LR), seed=U.SEED,
    )
print(f"\ntotal training time: {t_scratch.seconds:.1f}s")

In [ ]:
pred_scratch = scratch_model.predict_classes(X_test)
m = U.evaluate(y_test, pred_scratch, N_CLASSES)

results.append(U.RunResult(
    framework="Scratch (NumPy)", dataset=DATASET_SUB, model=ARCH,
    n_params=scratch_model.n_params(), epochs=EPOCHS,
    train_seconds=round(t_scratch.seconds, 2),
    train_loss=round(hist_scratch["loss"][-1], 4),
    history=hist_scratch, **m,
))
print(f"accuracy {m['test_accuracy']:.4f}   macro-F1 {m['f1_macro']:.4f}")

### What the first layer learned

With one input channel each kernel is a single 3×3 patch, so all sixteen render as small greyscale images.
`theory_notes.md` §2.2 is the point these pictures make: **nobody wrote these filters.** Their values are
parameters that backpropagation arrived at from random initialisation.

In [ ]:
Wk = scratch_model.layers[0].params["W"]        # (16, 1, 3, 3)
fig, axes = plt.subplots(2, 8, figsize=(9, 2.6))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(Wk[i, 0], cmap="RdBu_r")
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f"k{i}", fontsize=6)
fig.suptitle("Scratch CNN - first-layer 3x3 kernels (blue negative, red positive)")
fig.tight_layout(); plt.show()

---
## 4. Implementation B — TensorFlow / Keras

Keras is channels-last, so the arrays are transposed — a layout change only, not a different split.

> **Note on hardware.** Windows-native TensorFlow has shipped without GPU support since 2.10, so this leg
> runs on CPU while PyTorch uses the GPU where one is present. Parameter counts and accuracies compare
> across the rows; **wall-clock times do not.**

In [ ]:
U.set_seed(U.SEED)

# channels-first -> channels-last for Keras
Xtr_k = np.transpose(X_train, (0, 2, 3, 1))
Xte_k = np.transpose(X_test,  (0, 2, 3, 1))

def build_keras_cnn(name="mnist_cnn"):
    m = keras.Sequential([
        keras.layers.Input(shape=(H, W, C)),
        keras.layers.Conv2D(CH1, K, padding="same", activation="relu"),
        keras.layers.MaxPooling2D(2),
        keras.layers.Conv2D(CH2, K, padding="same", activation="relu"),
        keras.layers.MaxPooling2D(2),
        keras.layers.Flatten(),
        keras.layers.Dense(N_CLASSES),
    ], name=name)
    m.compile(optimizer=keras.optimizers.Adam(learning_rate=LR),
              loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=["accuracy"])
    return m

keras_model = build_keras_cnn()
keras_model.summary()
print(f"\nmatches hand count {EXPECTED_PARAMS:,}? {keras_model.count_params() == EXPECTED_PARAMS}")

In [ ]:
U.set_seed(U.SEED)
with U.Timer() as t_keras:
    hk = keras_model.fit(Xtr_k, y_train, validation_data=(Xte_k, y_test),
                         epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=2)
print(f"\ntotal training time: {t_keras.seconds:.1f}s")

In [ ]:
pred_keras = keras_model.predict(Xte_k, batch_size=256, verbose=0).argmax(axis=1)
m = U.evaluate(y_test, pred_keras, N_CLASSES)

hist_keras = {"loss": [float(v) for v in hk.history["loss"]],
              "val_acc": [float(v) for v in hk.history["val_accuracy"]]}

results.append(U.RunResult(
    framework="TensorFlow/Keras", dataset=DATASET_SUB, model=ARCH,
    n_params=int(keras_model.count_params()), epochs=EPOCHS,
    train_seconds=round(t_keras.seconds, 2),
    train_loss=round(hist_keras["loss"][-1], 4),
    history=hist_keras, **m,
))
print(f"accuracy {m['test_accuracy']:.4f}   macro-F1 {m['f1_macro']:.4f}")

---
## 5. Implementation C — PyTorch

$$\text{Zero Gradient} \rightarrow \text{Forward} \rightarrow \text{Loss} \rightarrow \text{Backward} \rightarrow \text{Update}$$

In [ ]:
U.set_seed(U.SEED)

class MnistCNN(nn.Module):
    def __init__(self, in_ch, n_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_ch, CH1, K, padding=PAD), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(CH1, CH2, K, padding=PAD),   nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(FLAT, n_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

torch_model = MnistCNN(C, N_CLASSES).to(DEVICE)
n_torch = sum(p.numel() for p in torch_model.parameters())
print(torch_model)
print(f"\nparameters: {n_torch:,}")
print(f"matches hand count {EXPECTED_PARAMS:,}? {n_torch == EXPECTED_PARAMS}")

In [ ]:
def train_torch(model, Xtr, ytr, Xte, yte, epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR):
    # The explicit five-step loop, reused for the subset and the full-data run.
    loader = DataLoader(TensorDataset(torch.tensor(Xtr), torch.tensor(ytr)),
                        batch_size=batch_size, shuffle=True)
    Xte_t = torch.tensor(Xte)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    hist = {"loss": [], "val_acc": []}

    with U.Timer() as t:
        for epoch in range(1, epochs + 1):
            model.train()
            running, nb = 0.0, 0
            for xb, yb in loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                optimizer.zero_grad()          # 1. zero gradient
                logits = model(xb)             # 2. forward
                loss = criterion(logits, yb)   # 3. loss
                loss.backward()                # 4. backward (autograd)
                optimizer.step()               # 5. update
                running += loss.item(); nb += 1

            model.eval()
            preds = []
            with torch.no_grad():
                for i in range(0, len(Xte_t), 512):
                    preds.append(model(Xte_t[i:i + 512].to(DEVICE)).argmax(1).cpu())
            acc = float((torch.cat(preds).numpy() == yte).mean())

            hist["loss"].append(running / nb)
            hist["val_acc"].append(acc)
            print(f"epoch {epoch}/{epochs}  loss {running/nb:.4f}  test_acc {acc:.4f}")

    return hist, t.seconds


def predict_torch(model, X):
    model.eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(X), 512):
            out.append(model(torch.tensor(X[i:i + 512]).to(DEVICE)).argmax(1).cpu())
    return torch.cat(out).numpy()


U.set_seed(U.SEED)
hist_torch, secs_torch = train_torch(torch_model, X_train, y_train, X_test, y_test)
print(f"\ntotal training time: {secs_torch:.1f}s")

In [ ]:
pred_torch = predict_torch(torch_model, X_test)
m = U.evaluate(y_test, pred_torch, N_CLASSES)

results.append(U.RunResult(
    framework="PyTorch", dataset=DATASET_SUB, model=ARCH,
    n_params=n_torch, epochs=EPOCHS,
    train_seconds=round(secs_torch, 2),
    train_loss=round(hist_torch["loss"][-1], 4),
    history=hist_torch, **m,
))
print(f"accuracy {m['test_accuracy']:.4f}   macro-F1 {m['f1_macro']:.4f}")

---
## 6. Scaling up — the full 60,000

Same architecture, all 60,000 training digits, Keras and PyTorch only.

In [ ]:
X_tr_f, y_tr_f, X_te_f, y_te_f, meta_f = U.load_mnist()
DATASET_FULL = "MNIST (full 60k)"
print(f"\ntrain {X_tr_f.shape}   test {X_te_f.shape}")
print(f"memory: train {X_tr_f.nbytes / 1e9:.2f} GB, test {X_te_f.nbytes / 1e9:.2f} GB")

In [ ]:
U.set_seed(U.SEED)
torch_full = MnistCNN(C, N_CLASSES).to(DEVICE)
hist_tf, secs_tf = train_torch(torch_full, X_tr_f, y_tr_f, X_te_f, y_te_f)

pred = predict_torch(torch_full, X_te_f)
m = U.evaluate(y_te_f, pred, N_CLASSES)
full_results.append(U.RunResult(
    framework="PyTorch", dataset=DATASET_FULL, model=ARCH,
    n_params=sum(p.numel() for p in torch_full.parameters()), epochs=EPOCHS,
    train_seconds=round(secs_tf, 2), train_loss=round(hist_tf["loss"][-1], 4),
    history=hist_tf, **m,
))
print(f"\naccuracy {m['test_accuracy']:.4f}   macro-F1 {m['f1_macro']:.4f}")

In [ ]:
U.set_seed(U.SEED)
keras_full = build_keras_cnn("mnist_cnn_full")
Xtr_kf = np.transpose(X_tr_f, (0, 2, 3, 1))
Xte_kf = np.transpose(X_te_f, (0, 2, 3, 1))

with U.Timer() as t_kf:
    hkf = keras_full.fit(Xtr_kf, y_tr_f, validation_data=(Xte_kf, y_te_f),
                         epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=2)

pred = keras_full.predict(Xte_kf, batch_size=256, verbose=0).argmax(axis=1)
m = U.evaluate(y_te_f, pred, N_CLASSES)
hist_kf = {"loss": [float(v) for v in hkf.history["loss"]],
           "val_acc": [float(v) for v in hkf.history["val_accuracy"]]}
full_results.append(U.RunResult(
    framework="TensorFlow/Keras", dataset=DATASET_FULL, model=ARCH,
    n_params=int(keras_full.count_params()), epochs=EPOCHS,
    train_seconds=round(t_kf.seconds, 2), train_loss=round(hist_kf["loss"][-1], 4),
    history=hist_kf, **m,
))
print(f"\naccuracy {m['test_accuracy']:.4f}   macro-F1 {m['f1_macro']:.4f}")

---
## 7. Comparison

### 7.1 The numbers slide 29 asks for

In [ ]:
table = U.results_table(results + full_results)
display(table)

print(f"\nall parameter counts identical: {table['n_params'].nunique() == 1} "
      f"({table['n_params'].iloc[0]:,})")
sub = table[table["dataset"] == DATASET_SUB]["test_accuracy"]
print(f"framework spread on the 10k subset: {sub.max() - sub.min():.4f} accuracy")

In [ ]:
U.plot_history({f"{r.framework} [{r.dataset}]": r.history for r in results + full_results},
               title="MNIST - one CNN, three implementations")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))
for ax, r in zip(axes, results):
    U.plot_confusion(r.confusion, CLASSES, title=r.framework, ax=ax)
fig.suptitle(f"Confusion matrices, {DATASET_SUB} (row-normalised)")
fig.tight_layout(); plt.show()

In [ ]:
# Which digits actually get confused, on the best model.
best = max(results + full_results, key=lambda r: r.test_accuracy)
cm = np.array(best.confusion, dtype=float)
np.fill_diagonal(cm, 0)
pairs = [(CLASSES[i], CLASSES[j], int(cm[i, j]))
         for i in range(N_CLASSES) for j in range(N_CLASSES) if cm[i, j] > 0]
pairs.sort(key=lambda t: -t[2])

print(f"most frequent confusions - {best.framework} on {best.dataset} "
      f"(accuracy {best.test_accuracy:.4f}):\n")
for true, pred_c, n in pairs[:8]:
    print(f"  a real {true} called {pred_c}: {n:>4} times")

### 7.2 Reproducing `02_mnist.ipynb`

This is what MNIST buys that the Intel data could not have. The repository root already ran **this exact
architecture, on this exact subset, with this exact seed**, and committed the result to
`results/02_mnist.json`. Rerunning it here is a genuine reproduction check.

Two outcomes are informative, and neither is a failure:

- **Parameter counts must match exactly.** They are determined by the architecture alone. Any difference
  means the two notebooks are not building the same model.
- **Accuracies should match closely, but need not be identical.** Library versions, CUDA kernels and
  non-deterministic GPU reductions all move the last digits. What matters is whether the gap is smaller
  than the seed-only noise floor that `04_compare.ipynb` measured at **0.053** accuracy.

In [ ]:
ref_path = os.path.join(D.ROOT, "results", "02_mnist.json")
if not os.path.exists(ref_path):
    print("results/02_mnist.json not found - skipping the reproduction check")
else:
    with open(ref_path, encoding="utf-8") as f:
        ref = json.load(f)
    ref_rows = {(r["framework"], r["dataset"]): r
                for r in ref["results"] + ref.get("full_results", [])}

    comp = []
    for r in results + full_results:
        old = ref_rows.get((r.framework, r.dataset))
        if old is None:
            continue
        comp.append({
            "framework": r.framework,
            "dataset": r.dataset,
            "params (02_mnist)": old["n_params"],
            "params (here)": r.n_params,
            "params match": old["n_params"] == r.n_params,
            "acc (02_mnist)": old["test_accuracy"],
            "acc (here)": r.test_accuracy,
            "difference": r.test_accuracy - old["test_accuracy"],
        })
    comp = pd.DataFrame(comp)
    display(comp.round(4))

    print(f"every parameter count reproduced exactly: {bool(comp['params match'].all())}")
    worst = comp["difference"].abs().max()
    print(f"largest accuracy difference: {worst:.4f}")
    print(f"seed-only noise floor from 04_compare.ipynb: 0.0530")
    print("-> within noise" if worst < 0.053
          else "-> LARGER than the noise floor; worth investigating")

---
## 8. Where each component lives

In [ ]:
component_table = pd.DataFrame([
    ["Data loading",     "ass4_utils.load_mnist -> NumPy", "same arrays (transposed to NHWC)", "same arrays"],
    ["Preprocessing",    "ass4_utils (shared)",            "shared",                     "shared"],
    ["Tensor layout",    "N,C,H,W",                        "N,H,W,C (channels-last)",    "N,C,H,W"],
    ["Model definition", "S.Sequential([...])",            "keras.Sequential([...])",    "nn.Module"],
    ["Convolution",      "S.Conv2D (im2col by hand)",      "keras.layers.Conv2D",        "nn.Conv2d"],
    ["Pooling",          "S.MaxPool2D (argmax routing)",   "keras.layers.MaxPooling2D",  "nn.MaxPool2d"],
    ["Activation",       "S.ReLU (mask multiply)",         "activation='relu'",          "nn.ReLU"],
    ["Flatten",          "S.Flatten",                      "keras.layers.Flatten",       "nn.Flatten"],
    ["Loss",             "S.SoftmaxCrossEntropy",          "SparseCategoricalCrossentropy", "nn.CrossEntropyLoss"],
    ["Gradient",         "hand-derived backward()",        "automatic (GradientTape)",   "automatic (autograd)"],
    ["Optimizer",        "S.Adam (written out)",           "keras.optimizers.Adam",      "torch.optim.Adam"],
    ["Training loop",    "explicit for-loop",              "model.fit()",                "explicit for-loop"],
], columns=["Component", "Scratch", "TensorFlow/Keras", "PyTorch"])

display(component_table)

In [ ]:
payload = {
    "meta": {k: v for k, v in meta.items() if k != "feature_names"},
    "subset": {"n_train": int(N_TRAIN_SUB), "n_test": int(N_TEST_SUB)},
    "results": [r.__dict__ for r in results],
    "full_results": [r.__dict__ for r in full_results],
    "component_table": component_table.to_dict(orient="records"),
    "substitution_note": (
        "MNIST stands in for Intel Image Classification, whose labelled seg_train/seg_test "
        "splits are absent from data/ - see 00_inventory.ipynb."
    ),
}
path = os.path.join(RESULTS, "n03_mnist_cnn.json")
with open(path, "w", encoding="utf-8") as f:
    json.dump(payload, f, indent=2, default=str)
print("saved", os.path.relpath(path, D.HERE))

## Saving the trained models

Five networks were trained: three on the subset for the framework comparison, two on the full 60,000.

In [ ]:
TAG = "n03_mnist_cnn"
by_key = {(r.framework, r.dataset): r for r in results + full_results}

for obj, fw, ds, stem in [
    (scratch_model, "Scratch (NumPy)",  DATASET_SUB,  "mnist_cnn_scratch_subset"),
    (keras_model,   "TensorFlow/Keras", DATASET_SUB,  "mnist_cnn_keras_subset"),
    (torch_model,   "PyTorch",          DATASET_SUB,  "mnist_cnn_torch_subset"),
    (keras_full,    "TensorFlow/Keras", DATASET_FULL, "mnist_cnn_keras_full"),
    (torch_full,    "PyTorch",          DATASET_FULL, "mnist_cnn_torch_full"),
]:
    r = by_key[(fw, ds)]
    U.save_model(obj, stem, TAG, architecture=r.model, dataset=r.dataset,
                 epochs=r.epochs, test_accuracy=r.test_accuracy, f1_macro=r.f1_macro,
                 classes=CLASSES)

display(U.list_models(TAG)[["name", "framework", "n_params", "dataset", "test_accuracy"]])

---
## 9. What this dataset shows

**The three implementations agree — third dataset, third confirmation.** Identical parameter counts to the
digit, identical tensor shapes at every stage, accuracies separated by less than the seed-only noise floor.
`S.Conv2D`, `keras.layers.Conv2D` and `nn.Conv2d` are three names for one function.

**And the experiment reproduces across machines.** §7.2 reruns a committed result from `02_mnist.ipynb`
and compares. Parameter counts are determined by the architecture and must match exactly; accuracies drift
in the last digits because GPU reductions are not bit-deterministic. A reproduction that lands inside the
measured noise band is a pass, and saying so precisely — rather than expecting identical floats — is the
honest way to report it.

**One channel is not a special case.** MNIST is the only `C = 1` dataset here, and nothing in any of the
three implementations needed changing for it. The kernel window simply flattens to 9 values instead of 27.

**Greyscale digits are an easier problem than colour photographs, and much easier than natural scenes.**
Expect high nineties. That is worth remembering when reading notebook 04: MNIST, like the rice images, sits
close enough to the ceiling that architectural improvements have little room to demonstrate anything, which
is exactly why that notebook trains on a deliberately small slice.

**Finally, why this notebook exists at all.** The Intel scene dataset arrived without its labels.
Substituting MNIST kept the assignment's structure intact — one tabular dataset, two image datasets, three
implementations each — without inventing numbers from data that cannot support them.